# COMP6011 Task 2 — Experiment 1: LLM Benchmarking

This notebook evaluates multiple open-weight large language models for suicide risk classification using a structured prompt-based approach.

Models evaluated:
- Qwen2.5 7B Instruct
- Mistral 7B Instruct
- Llama 3.1 8B Instruct

The task is to classify conversational dialogue into five clinically relevant risk categories:
safe, indicator, ideation, behavior, attempt

## 1. Objective

The objective of this experiment is to compare the performance of multiple large language models in detecting suicide risk from conversational data.

The evaluation focuses on:
- Classification accuracy
- Macro precision, recall, and F1-score
- Output reliability
- Ability to handle nuanced emotional language

In [ ]:
!nvidia-smi

Thu Apr  9 08:30:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Environment Setup

This section installs required dependencies for running transformer-based models and evaluation metrics.

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece pandas openpyxl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00


## 3. Library Imports and Hardware Configuration

This section loads required libraries and verifies GPU availability for efficient model inference.

In [ ]:
from getpass import getpass
HF_TOKEN = getpass("Enter your Hugging Face token: ")

Enter your Hugging Face token: ··········


## 4. Dataset Upload and Inspection

The dataset consists of conversational dialogue cases annotated across five suicide risk categories.

The dataset is manually uploaded and inspected to verify structure and column names.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving student_assignment_10_cases.xlsx to student_assignment_10_cases.xlsx


## 5. Data Preprocessing

This section standardises column names and ensures consistency in label formatting for evaluation.

In [ ]:
import pandas as pd

FILE_PATH = "student_assignment_10_cases.xlsx"
SHEET_NAME = "Assignment_Cases"

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df.columns = [c.strip() for c in df.columns]

print(df.head())
print(df.columns.tolist())
print(df["Risk Level"].value_counts())

   Case ID                               Paraphrased Dialogue Risk Level
0        1  User: Everything is just too much right now. I...    attempt
1        2  User: I have nothing left to stay for. I’ve fe...    attempt
2        3  User: Time doesn't feel real anymore. I have n...   behavior
3        4  User: I think I’m nearing the end of the road....   behavior
4        5  User: Just make it stop. Please. I just want t...   ideation
['Case ID', 'Paraphrased Dialogue', 'Risk Level']
Risk Level
attempt      2
behavior     2
ideation     2
indicator    2
safe         2
Name: count, dtype: int64


## 6. Risk Categories

The classification task uses five predefined categories:

- **safe**: No indication of suicide risk  
- **indicator**: General distress or warning signs  
- **ideation**: Suicidal thoughts without action  
- **behavior**: Planning or preparatory actions  
- **attempt**: Evidence of suicide attempt  

These categories represent increasing levels of risk.

In [ ]:
LABELS = ["safe", "indicator", "ideation", "behavior", "attempt"]

def normalize_label(label):
    if pd.isna(label):
        return None
    label = str(label).strip().lower()
    mapping = {
        "behaviour": "behavior"
    }
    return mapping.get(label, label)

df["true_label"] = df["Risk Level"].apply(normalize_label)

In [ ]:
QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
LLAMA_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

## 7. Output Processing and Label Extraction

Model outputs are expected in JSON format. However, due to inconsistencies in generation, a fallback extraction method is implemented.

This ensures:
- Robust label detection
- Reduced loss of predictions
- Improved evaluation reliability

The following implementation defines the model loading and quantisation setup used for efficient inference.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=HF_TOKEN,
        trust_remote_code=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    return tokenizer, model

## 8. Prompt Engineering

A structured prompt is used to guide the model toward consistent classification.

The prompt includes:
- Clear task definition
- Category descriptions
- Strict JSON output requirement

This improves both accuracy and output reliability.

In [ ]:
SYSTEM_PROMPT = """
You are a strict classification system.

Task:
Classify the dialogue into ONE of these labels:
safe, indicator, ideation, behavior, attempt

Output MUST be valid JSON ONLY.

Correct format:
{"label": "safe", "reason": "brief explanation"}

Rules:
- Output ONLY JSON
- No markdown
- No extra text
- Label must be exactly one of the five options
"""

def build_prompt(dialogue: str) -> str:
    return f"""
{SYSTEM_PROMPT}

Dialogue:
{dialogue}
""".strip()

## 9. Output Processing and Label Extraction

Model outputs are expected in JSON format. However, due to inconsistencies in generation, a fallback extraction method is implemented.

This ensures:
- Robust label detection  
- Reduced loss of predictions  
- Improved evaluation reliability  

In [ ]:
import json
import re

def extract_json_label(text: str):
    if not text:
        return None, None, text

    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group(0))
            label = normalize_label(obj.get("label"))
            reason = obj.get("reason", "")
            if label in LABELS:
                return label, reason, text
        except:
            pass

    text_lower = text.lower()
    for label in LABELS:
        if label in text_lower:
            return label, None, text

    return None, None, text

def classify_dialogue_local(tokenizer, model, dialogue: str, max_new_tokens: int = 120):
    prompt = build_prompt(dialogue)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            pad_token_id=tokenizer.eos_token_id
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = full_text[len(prompt):].strip() if full_text.startswith(prompt) else full_text

    label, reason, raw = extract_json_label(generated_text)
    return {
        "label": label,
        "reason": reason,
        "raw_output": raw
    }

## 10. Inference Pipeline

The above functions together form the inference pipeline used in this experiment.

This pipeline performs:
- Prompt construction  
- Model inference  
- Output decoding  
- Label extraction  

This ensures that raw model outputs are transformed into structured predictions suitable for evaluation.

## 9. Model Configuration

Three instruction-tuned large language models are evaluated in this study:

- Qwen2.5 7B Instruct  
- Mistral 7B Instruct  
- Llama 3.1 8B Instruct  

These models are selected due to their strong performance in natural language understanding tasks and their availability as open-weight models.

To enable efficient inference within limited computational resources, all models are loaded using 4-bit quantisation. This significantly reduces memory usage while maintaining acceptable performance.

Due to hardware constraints, models are evaluated sequentially, with memory cleared between runs to prevent out-of-memory errors.

In [ ]:
results = df[["Case ID", "Paraphrased Dialogue", "true_label"]].copy()
results["qwen_pred"] = None
results["qwen_reason"] = None
results["qwen_raw"] = None
results["llama_pred"] = None
results["llama_reason"] = None
results["llama_raw"] = None

## 10. Inference Pipeline

This section defines the classification pipeline used to generate predictions for each model.

The pipeline consists of the following steps:

- Construction of a structured prompt using the input dialogue  
- Tokenisation of the prompt for model input  
- Model inference using deterministic generation  
- Decoding of the generated output  
- Extraction of the predicted label using the output processing function  

This pipeline ensures that raw model outputs are transformed into structured predictions suitable for quantitative evaluation.

## 11. Model Evaluation — Qwen2.5 7B Instruct

This section evaluates Qwen on the dataset and records predictions and performance metrics.

In [ ]:
qwen_tokenizer, qwen_model = load_model(QWEN_MODEL)

for i, row in results.iterrows():
    out = classify_dialogue_local(qwen_tokenizer, qwen_model, row["Paraphrased Dialogue"])
    results.at[i, "qwen_pred"] = out["label"]
    results.at[i, "qwen_reason"] = out["reason"]
    results.at[i, "qwen_raw"] = out["raw_output"]
    print(f"Qwen done: case {row['Case ID']} -> {out['label']}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen done: case 1 -> attempt
Qwen done: case 2 -> attempt
Qwen done: case 3 -> attempt
Qwen done: case 4 -> attempt
Qwen done: case 5 -> attempt
Qwen done: case 6 -> ideation
Qwen done: case 7 -> attempt
Qwen done: case 8 -> attempt
Qwen done: case 9 -> safe
Qwen done: case 10 -> ideation


In [ ]:
import torch, gc
del qwen_model
del qwen_tokenizer
gc.collect()
torch.cuda.empty_cache()

## 12. Model Evaluation — Mistral 7B Instruct

This section evaluates Mistral 7B Instruct and compares its performance with other models. Mistral is expected to provide efficient inference with competitive accuracy, but may exhibit inconsistencies in output formatting.

In [ ]:
MISTRAL_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

In [ ]:
results["mistral_pred"] = None
results["mistral_reason"] = None
results["mistral_raw"] = None

In [ ]:
mistral_tokenizer, mistral_model = load_model(MISTRAL_MODEL)

for i, row in results.iterrows():
    out = classify_dialogue_local(mistral_tokenizer, mistral_model, row["Paraphrased Dialogue"])
    results.at[i, "mistral_pred"] = out["label"]
    results.at[i, "mistral_reason"] = out["reason"]
    results.at[i, "mistral_raw"] = out["raw_output"]
    print(f"Mistral done: case {row['Case ID']} -> {out['label']}")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral done: case 1 -> None
Mistral done: case 2 -> None
Mistral done: case 3 -> None
Mistral done: case 4 -> safe
Mistral done: case 5 -> ideation
Mistral done: case 6 -> None
Mistral done: case 7 -> ideation
Mistral done: case 8 -> ideation
Mistral done: case 9 -> safe
Mistral done: case 10 -> safe


## 13. Model Evaluation — Llama 3.1 8B Instruct

This section evaluates Llama 3.1 8B Instruct. Due to access restrictions and high computational requirements, the model may produce incomplete or inconsistent outputs. This reflects real-world deployment constraints and is considered in the analysis.

In [ ]:
llama_tokenizer, llama_model = load_model(LLAMA_MODEL)

for i, row in results.iterrows():
    out = classify_dialogue_local(llama_tokenizer, llama_model, row["Paraphrased Dialogue"])
    results.at[i, "llama_pred"] = out["label"]
    results.at[i, "llama_reason"] = out["reason"]
    results.at[i, "llama_raw"] = out["raw_output"]
    print(f"Llama done: case {row['Case ID']} -> {out['label']}")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Llama done: case 1 -> None
Llama done: case 2 -> ideation
Llama done: case 3 -> None
Llama done: case 4 -> ideation
Llama done: case 5 -> ideation
Llama done: case 6 -> None
Llama done: case 7 -> None
Llama done: case 8 -> ideation
Llama done: case 9 -> safe
Llama done: case 10 -> ideation


## 14. Model Evaluation Results

The following results present the performance of each model based on classification accuracy, macro precision, macro recall, and macro F1-score. These metrics provide a balanced evaluation across all risk categories.

Confusion matrices and classification reports are also included to analyse how each model performs across different classes.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

def evaluate_model(y_true, y_pred, model_name):
    valid_pairs = [(t, p) for t, p in zip(y_true, y_pred) if p in LABELS]
    y_true_clean = [t for t, p in valid_pairs]
    y_pred_clean = [p for t, p in valid_pairs]

    acc = accuracy_score(y_true_clean, y_pred_clean)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_clean,
        y_pred_clean,
        labels=LABELS,
        average="macro",
        zero_division=0
    )
    cm = confusion_matrix(y_true_clean, y_pred_clean, labels=LABELS)

    print(f"\n=== {model_name} ===")
    print(f"Accuracy: {acc:.3f}")
    print(f"Macro Precision: {precision:.3f}")
    print(f"Macro Recall: {recall:.3f}")
    print(f"Macro F1: {f1:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_true_clean, y_pred_clean, labels=LABELS, zero_division=0))
    print("Confusion Matrix:")
    print(cm)

    return {
        "Model": model_name,
        "Accuracy": acc,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1
    }, cm

## 15. Evaluation Outputs

This section presents the detailed outputs generated by each model, including predicted labels and corresponding reasoning.

These outputs provide insight into model behaviour and support qualitative analysis of classification performance, particularly in cases where predictions are incorrect or ambiguous.

In [ ]:
qwen_metrics, qwen_cm = evaluate_model(
    results["true_label"].tolist(),
    results["qwen_pred"].tolist(),
    "Qwen2.5 7B Instruct"
)

mistral_metrics, mistral_cm = evaluate_model(
    results["true_label"].tolist(),
    results["mistral_pred"].tolist(),
    "Mistral 7B Instruct"
)


=== Qwen2.5 7B Instruct ===
Accuracy: 0.400
Macro Precision: 0.357
Macro Recall: 0.400
Macro F1: 0.322

Classification Report:
              precision    recall  f1-score   support

        safe       1.00      0.50      0.67         2
   indicator       0.00      0.00      0.00         2
    ideation       0.50      0.50      0.50         2
    behavior       0.00      0.00      0.00         2
     attempt       0.29      1.00      0.44         2

    accuracy                           0.40        10
   macro avg       0.36      0.40      0.32        10
weighted avg       0.36      0.40      0.32        10

Confusion Matrix:
[[1 0 1 0 0]
 [0 0 0 0 2]
 [0 0 1 0 1]
 [0 0 0 0 2]
 [0 0 0 0 2]]

=== Mistral 7B Instruct ===
Accuracy: 0.500
Macro Precision: 0.200
Macro Recall: 0.400
Macro F1: 0.260

Classification Report:
              precision    recall  f1-score   support

        safe       0.67      1.00      0.80         2
   indicator       0.00      0.00      0.00         2
    idea

In [ ]:
llama_metrics, llama_cm = evaluate_model(
    results["true_label"].tolist(),
    results["llama_pred"].tolist(),
    "Llama 3.1 8B Instruct"
)


=== Llama 3.1 8B Instruct ===
Accuracy: 0.333
Macro Precision: 0.240
Macro Recall: 0.300
Macro F1: 0.200

Classification Report:
              precision    recall  f1-score   support

        safe       1.00      0.50      0.67         2
   indicator       0.00      0.00      0.00         1
    ideation       0.20      1.00      0.33         1
    behavior       0.00      0.00      0.00         1
     attempt       0.00      0.00      0.00         1

    accuracy                           0.33         6
   macro avg       0.24      0.30      0.20         6
weighted avg       0.37      0.33      0.28         6

Confusion Matrix:
[[1 0 1 0 0]
 [0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 1 0 0]]


## 17. Export Results

The results are exported as CSV files for inclusion in the report and appendix. These files contain case-level predictions and model outputs, enabling reproducibility and supporting detailed analysis outside the notebook.

In [ ]:
results.to_csv("task2_case_predictions.csv", index=False)
print("Saved predictions file")

Saved predictions file


## 18. Key Observations

- Qwen2.5 demonstrates the most balanced performance across all metrics, with consistent and valid outputs.
- Mistral achieves higher accuracy but shows lower precision, indicating less reliable classification across categories.
- Llama exhibits reduced performance and produces several invalid or missing outputs, highlighting limitations in constrained environments.
- Misclassifications are more frequent in ambiguous cases, particularly between indicator and ideation categories.
- Structured prompting significantly improves output consistency and reduces invalid predictions.

These findings highlight that in safety-critical applications, output reliability is as important as accuracy, and models must be evaluated based on both performance and consistency.